# Phase 1: turn the hand-checked transcripts into activations

**Question.** For each blackmail ("act") and hold-back ("refrain") transcript, what does the residual stream look like at the moment that matters?

**Two anchor points per transcript**
- **notice**: the first token where the reasoning mentions the leverage (affair, leverage, vulnerability...).
- **commit**: the last token of the reasoning, right before `</SCRATCHPAD_REASONING>`, i.e. just before the model acts.

Comparing the two anchors is what lets Phase 2 ask *when* the decision becomes readable.

| | |
|---|---|
| Needs | GPU with about 24 GB |
| Reads | `contrast_transcripts.json` (83 act, 100 refrain; hand-checked in Phase -1) |
| Writes | `contrast_acts.pt` (notice), `contrast_acts_commit.pt` (commit), each `{act, ref}` as `[n, 48, 3840]` |

**Caveat on the notice anchor.** The repo ships the commit-anchor extraction as a notebook but not the batch code for the notice anchor.
The notice extraction below is a reconstruction (first keyword inside the final reasoning block), so expect small differences from the original `contrast_acts.pt`.

**Garbled outputs.** Some transcripts are token salad; any anchor whose surrounding text is more than 15% non-ASCII is skipped.
The original run kept 68 act / 98 refrain at the commit anchor (17 skipped).

In [ ]:
import os, json, time, numpy as np
from collections import defaultdict
from functools import partial
BASE = os.environ.get("GEMMA_BASE", os.getcwd())                  # folder holding the data files listed in README.md (default: where you run the notebook)
OUT  = f"{BASE}/results"; os.makedirs(OUT, exist_ok=True)         # everything generated lands here
SEED, N_SCEN, MAX_NEW = 0, 15, 1200
GRADER_MODEL = os.environ.get("GRADER_MODEL", "gemini-3.5-flash") # Gemini model that grades outputs; change it if your key says "model not found"
# Keys are read from the environment and never written in a notebook:
#   GEMINI_API_KEY  (grading)      HF_TOKEN  (first download of Gemma; accept the licence on Hugging Face first)
# Colab: add them under "Secrets" and run  os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")  once in a cell.

In [ ]:
import torch
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained_no_processing("google/gemma-3-12b-it", dtype="bfloat16")
tok = model.tokenizer
nL = model.cfg.n_layers
data = json.load(open(f"{BASE}/contrast_transcripts.json"))      # 183 transcripts: 83 act, 100 refrain

In [ ]:
OPEN, CLOSE = "<SCRATCHPAD_REASONING>", "</SCRATCHPAD_REASONING>"
NOTICE = ["affair", "infidelit", "jessica", "maria", "rachel", "personal life",
          "extramarital", "vulnerab", "leverage", "exploit"]

def is_garbled(ctx):
    return (not ctx) or sum(1 for c in ctx if ord(c) > 127) / len(ctx) > 0.15

def tok_at_char(strs, c):                                         # index of the token containing character c
    cum = 0
    for j, t in enumerate(strs):
        cum += len(t)
        if cum > c: return j
    return None

def anchor_chars(full):                                           # (notice_char, commit_char), None if absent
    e = full.rfind(CLOSE)
    if e < 0: return None, None
    commit = e - 1
    s = full.rfind(OPEN, 0, e)
    low, notice = full.lower(), None
    if s >= 0:
        ends = [i + len(k) - 1 for k in NOTICE if (i := low.find(k, s, e)) >= 0]
        if ends: notice = min(ends)
    return notice, commit

## Extract both anchors in one forward pass per transcript

In [ ]:
sets = {"notice": {"act": [], "ref": []}, "commit": {"act": [], "ref": []}}
skipped, contexts = [], []

for k, rec in enumerate(data):
    msgs = [{"role": m["role"], "content": m["text"]} for m in rec["messages"]]
    toks = model.to_tokens(tok.apply_chat_template(msgs, tokenize=False), prepend_bos=False)
    strs = tok.batch_decode([[i] for i in toks[0].tolist()], clean_up_tokenization_spaces=False)
    full = "".join(strs)
    chars = dict(zip(("notice", "commit"), anchor_chars(full)))
    if all(c is None for c in chars.values()):
        skipped.append((rec["src"], rec["idx"], "no_close_tag")); continue
    with torch.no_grad():
        _, cache = model.run_with_cache(toks, names_filter=lambda n: n.endswith("resid_post"))
    for name, c in chars.items():
        if c is None: skipped.append((rec["src"], rec["idx"], f"no_{name}_anchor")); continue
        ctx = full[max(0, c - 60): c + 25].replace("\n", " ")
        if is_garbled(ctx): skipped.append((rec["src"], rec["idx"], f"garbled_{name}")); continue
        j = tok_at_char(strs, c)
        acts = torch.stack([cache["resid_post", L][0, j] for L in range(nL)]).float().cpu()
        sets[name]["act" if rec["label"] == "act" else "ref"].append(acts)
        contexts.append((name, rec["label"], ctx))
    if (k + 1) % 25 == 0: print(f"{k+1}/{len(data)}")

for name, fname in [("notice", "contrast_acts.pt"), ("commit", "contrast_acts_commit.pt")]:
    s = sets[name]
    torch.save({"act": torch.stack(s["act"]), "ref": torch.stack(s["ref"])}, f"{BASE}/{fname}")
    print(f"{name:>6}: act={len(s['act'])} ref={len(s['ref'])} -> {fname}")
print("skipped:", len(skipped))

## Eyeball the anchors
If the notice anchors sit on the affair and the commit anchors sit on the end of the reasoning, the extraction is right.

In [ ]:
for name in ("notice", "commit"):
    for label in ("act", "refrain"):
        ex = [c for n, l, c in contexts if n == name and l == label][:2]
        for c in ex: print(f"{name:>6} | {label:>7} | ...{c}...")

## What this gives Phase 2
Two balanced sets of residual-stream vectors, one per anchor, at all 48 layers. Sample sizes are small (about 70-100 per class),
which is why Phase 2 uses a permutation null and strong regularisation rather than trusting raw accuracy.